# 06B_MODIS_Feature_Extraction.ipynb
Extract MODIS Land Surface Temperature (LST) features from image_metadata.csv.

In [ ]:
!pip -q install earthengine-api pandas tqdm

import ee
import pandas as pd
from tqdm import tqdm
from google.colab import files

ee.Authenticate()
ee.Initialize(project='heatwave-flood-prediction')

print("Upload image_metadata.csv")
uploaded = files.upload()
fname = list(uploaded.keys())[0]

df = pd.read_csv(fname)

results = []
skipped = []

for _, row in tqdm(df.iterrows(), total=len(df)):

    event_id = row['event_id']
    date = ee.Date(str(row['image_date']))
    point = ee.Geometry.Point([float(row['longitude']), float(row['latitude'])])

    collection = (
        ee.ImageCollection('MODIS/061/MOD11A1')
        .filterBounds(point)
        .filterDate(date.advance(-3, 'day'), date.advance(4, 'day'))
        .sort('system:time_start')
    )

    if collection.size().getInfo() == 0:
        skipped.append({
            'event_id': event_id,
            'reason': 'No MODIS image found'
        })
        continue

    image = collection.first()

    try:
        vals = image.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=point,
            scale=1000,
            maxPixels=1e9
        ).getInfo()

        day = vals.get('LST_Day_1km')
        night = vals.get('LST_Night_1km')

        results.append({
            'event_id': event_id,
            'image_date': row['image_date'],
            'LST_Day_C': None if day is None else day * 0.02 - 273.15,
            'LST_Night_C': None if night is None else night * 0.02 - 273.15
        })

    except Exception as e:
        skipped.append({
            'event_id': event_id,
            'reason': str(e)
        })

pd.DataFrame(results).to_csv('modis_features.csv', index=False)
pd.DataFrame(skipped).to_csv('modis_skipped.csv', index=False)

files.download('modis_features.csv')
files.download('modis_skipped.csv')

print("Done!")
